# Schema Exploration
Goal: understand the columns in each dataset, check ID alignment between static GTFS and real-time feeds, and confirm the join is viable before building a training dataset.

In [ ]:
import io
import json
import zipfile
from pathlib import Path

import pandas as pd

DATA_DIR = Path('../data/raw')
GTFS_ZIP = DATA_DIR / 'google_transit.zip'

## 1. Load all core GTFS tables

In [ ]:
with zipfile.ZipFile(GTFS_ZIP, 'r') as z:
    routes     = pd.read_csv(io.BytesIO(z.read('routes.txt')))
    stops      = pd.read_csv(io.BytesIO(z.read('stops.txt')))
    trips      = pd.read_csv(io.BytesIO(z.read('trips.txt')))
    stop_times = pd.read_csv(io.BytesIO(z.read('stop_times.txt')))
    calendar   = pd.read_csv(io.BytesIO(z.read('calendar.txt'))) if 'calendar.txt' in z.namelist() else None
    calendar_dates = pd.read_csv(io.BytesIO(z.read('calendar_dates.txt'))) if 'calendar_dates.txt' in z.namelist() else None

print('Loaded tables:')
print(f'  routes:         {len(routes):,} rows')
print(f'  stops:          {len(stops):,} rows')
print(f'  trips:          {len(trips):,} rows')
print(f'  stop_times:     {len(stop_times):,} rows')
if calendar is not None:
    print(f'  calendar:       {len(calendar):,} rows')
if calendar_dates is not None:
    print(f'  calendar_dates: {len(calendar_dates):,} rows')

## 2. trips.txt — columns, sample, and service_id

In [ ]:
print('=== trips.txt columns ===')
print(trips.dtypes)
print()
print('=== Sample rows ===')
trips.head(5)

In [ ]:
print('=== trip_id format (first 10) ===')
print(trips['trip_id'].head(10).tolist())
print()
print('=== service_id unique values (first 20) ===')
print(trips['service_id'].unique()[:20])
print()
print('=== direction_id value counts ===')
print(trips['direction_id'].value_counts())

## 3. stop_times.txt — columns, sample, and arrival_time format

In [ ]:
print('=== stop_times.txt columns ===')
print(stop_times.dtypes)
print()
print('=== Sample rows ===')
stop_times.head(5)

In [ ]:
print('=== arrival_time format (first 10) ===')
print(stop_times['arrival_time'].head(10).tolist())
print()
print('=== stop_sequence range ===')
print(stop_times['stop_sequence'].describe())
print()
# Check for any times > 24:00:00 (GTFS allows this for trips past midnight)
late_night = stop_times[stop_times['arrival_time'].str.startswith(('24:', '25:', '26:', '27:', '28:'), na=False)]
print(f'=== Times past midnight (hour >= 24): {len(late_night):,} rows ===')
if len(late_night) > 0:
    print(late_night['arrival_time'].head(5).tolist())

## 4. calendar.txt — how service_id maps to days of the week

In [ ]:
if calendar is not None:
    print('=== calendar.txt columns ===')
    print(calendar.dtypes)
    print()
    print('=== Sample rows ===')
    print(calendar.head(10).to_string())
else:
    print('No calendar.txt found — service days may be in calendar_dates.txt only')
    if calendar_dates is not None:
        print(calendar_dates.head(10).to_string())

## 5. Real-time feed — trip_updates schema and delay stats

In [ ]:
with open(DATA_DIR / 'realtime_trip_updates.json') as f:
    trip_updates = json.load(f)

entities = trip_updates.get('entity', [])
print(f'Total trip_update entities: {len(entities):,}')
print()
print('=== First entity (full structure) ===')
print(json.dumps(entities[0], indent=2))

In [ ]:
# Flatten all stop_time_updates into a dataframe
rows = []
for entity in entities:
    tu = entity.get('trip_update', {})
    trip = tu.get('trip', {})
    trip_id  = trip.get('trip_id')
    route_id = trip.get('route_id')
    for stu in tu.get('stop_time_update', []):
        arrival = stu.get('arrival') or {}
        rows.append({
            'trip_id':       trip_id,
            'route_id':      route_id,
            'stop_id':       stu.get('stop_id'),
            'stop_sequence': stu.get('stop_sequence'),
            'arrival_delay': arrival.get('delay'),
            'arrival_time':  arrival.get('time'),
        })

rt = pd.DataFrame(rows)
print(f'Total stop_time_update rows: {len(rt):,}')
print()
print('=== Columns and dtypes ===')
print(rt.dtypes)
print()
print('=== Sample rows ===')
rt.head(10)

In [ ]:
print('=== arrival_delay stats (seconds) ===')
print(rt['arrival_delay'].describe())
print()
null_delay = rt['arrival_delay'].isna().sum()
print(f'Rows with null arrival_delay: {null_delay:,} ({100*null_delay/len(rt):.1f}%)')
print()
print('=== Delay in minutes (non-null) ===')
delay_min = rt['arrival_delay'].dropna() / 60
print(f'  Min:    {delay_min.min():.1f} min')
print(f'  Max:    {delay_min.max():.1f} min')
print(f'  Median: {delay_min.median():.1f} min')
print(f'  Mean:   {delay_min.mean():.1f} min')
print()
late_pct = (rt['arrival_delay'].dropna() > 300).mean() * 100
print(f'% of stop events currently late > 5 min: {late_pct:.1f}%')

## 6. THE KEY CHECK — do trip_ids match between static and real-time?

In [ ]:
static_trip_ids = set(trips['trip_id'].astype(str))
rt_trip_ids     = set(rt['trip_id'].dropna().astype(str))

matched    = rt_trip_ids & static_trip_ids
rt_only    = rt_trip_ids - static_trip_ids
static_only = static_trip_ids - rt_trip_ids

print('=== trip_id overlap ===')
print(f'  Real-time trip_ids:          {len(rt_trip_ids):,}')
print(f'  Static trip_ids:             {len(static_trip_ids):,}')
print(f'  Matched (in both):           {len(matched):,}  <- want this to be high')
print(f'  In real-time only (no match):{len(rt_only):,}  <- these rows cannot be joined')
print(f'  Match rate:                  {100*len(matched)/len(rt_trip_ids):.1f}%')
print()
if rt_only:
    print('=== Sample unmatched real-time trip_ids ===')
    print(list(rt_only)[:10])
    print()
    print('=== Sample static trip_ids for comparison ===')
    print(list(static_trip_ids)[:10])

## 7. Bonus — do stop_ids match?

In [ ]:
static_stop_ids = set(stops['stop_id'].astype(str))
rt_stop_ids     = set(rt['stop_id'].dropna().astype(str))

stop_matched = rt_stop_ids & static_stop_ids
stop_rt_only = rt_stop_ids - static_stop_ids

print('=== stop_id overlap ===')
print(f'  Real-time stop_ids:          {len(rt_stop_ids):,}')
print(f'  Static stop_ids:             {len(static_stop_ids):,}')
print(f'  Matched:                     {len(stop_matched):,}')
print(f'  In real-time only:           {len(stop_rt_only):,}')
print(f'  Match rate:                  {100*len(stop_matched)/len(rt_stop_ids):.1f}%')

## Summary
After running all cells, answer these questions before moving on:
1. What is the `trip_id` match rate? (Cell 6) — must be high (>90%) to proceed
2. What is `arrival_delay` null rate? (Cell 5) — if >50% null, the delay signal is weak
3. What format is `arrival_time` in `stop_times.txt`? (Cell 3) — expect `HH:MM:SS` strings
4. Does `calendar.txt` exist and map `service_id` to weekdays? (Cell 4)